# AML - LEAVE ONE VARIANT OUT 
This script submits the commands for the leave-one-variant-out (LOVO) analysis. It performs Step 2 of the SAIGE-GENE+ association tests for each variant set, where each set has exactly one variant removed from the full collection. The script iterates over all annotation files, with each file representing a reduced variant set missing one specific variant. By looping through these files, the pipeline re-runs Step 2 for every leave-one-out subset, allowing for evaluation of each variant’s individual contribution to the overall gene-based association signal and identifying variants that disproportionately influence the test statistic.

In [1]:
# Imports the 'os' module, which allows interaction with the operating system (e.g., environment variables, file paths).
import os 

# Retrieves the value of the environment variable 'WORKSPACE_BUCKET'.
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Displays the value of 'my_bucket' (i.e., the workspace bucket URL or path).
my_bucket


'gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe'

# COMMAND 

In [3]:
%%writefile biosample_age_single_assoc.sh

#!/bin/bash
set -o errexit
set -o nounset

for pop in EUR; do
  for pheno in biosample_age; do
   
        echo "Processing pop: ${pop} | phenotype: ${pheno} | chr: ${chrom}" 
        
        step2_SPAtests.R \
            --bedFile="${plink_bed}" \
            --bimFile="${plink_bim}" \
            --famFile="${plink_fam}" \
            --SAIGEOutputFile="${output}/${pop}_${pheno}_single_assoc_chr${chrom}" \
            --LOCO=FALSE \
            --AlleleOrder=alt-first \
            --minMAF=0 \
            --minMAC=0.5 \
            --maxMAF=0.015 \
            --GMMATmodelFile="${null_glmm}/${pop}_${pheno}_null_glmm.rda" \
            --varianceRatioFile="${null_glmm}/${pop}_${pheno}_null_glmm.varianceRatio.txt" \
            --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
            --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
            --is_overwrite_output=TRUE \
            --is_Firth_beta=TRUE \
            --pCutoffforFirth=0.05 \
            --is_output_moreDetails=TRUE \
            --max_MAC_for_ER=10 


    done
done


Writing biosample_age_single_assoc.sh


In [4]:
!gsutil cp biosample_age_single_assoc.sh {my_bucket}/data/scripts/

Copying file://biosample_age_single_assoc.sh [Content-Type=text/x-sh]...
/ [1 files][  1.2 KiB/  1.2 KiB]                                                
Operation completed over 1 objects/1.2 KiB.                                      


In [5]:
!gsutil cat {my_bucket}/data/scripts/biosample_age_single_assoc.sh


#!/bin/bash
set -o errexit
set -o nounset

for pop in EUR; do
  for pheno in biosample_age; do
   
        echo "Processing pop: ${pop} | phenotype: ${pheno} | chr: ${chrom}" 
        
        step2_SPAtests.R \
            --bedFile="${plink_bed}" \
            --bimFile="${plink_bim}" \
            --famFile="${plink_fam}" \
            --SAIGEOutputFile="${output}/${pop}_${pheno}_single_assoc_chr${chrom}" \
            --LOCO=FALSE \
            --AlleleOrder=alt-first \
            --minMAF=0 \
            --minMAC=0.5 \
            --maxMAF=0.015 \
            --GMMATmodelFile="${null_glmm}/${pop}_${pheno}_null_glmm.rda" \
            --varianceRatioFile="${null_glmm}/${pop}_${pheno}_null_glmm.varianceRatio.txt" \
            --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
            --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
            --is_overwrite_o

# SUBMIT COMMAND 

In [6]:
%%bash --out saige_gene_step02

LOWER=2
UPPER=3
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bed" \
  --input plink_bim="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bim" \
  --input plink_fam="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/age_tests/" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/biosample_age_single_assoc.sh"

done

Job properties:
  job-id: biosample---vienose--260415-210859-47
  job-name: biosample-age-single-assoc
  user-id: vienose
Provider internal-id (operation): projects/terra-vpc-sc-fc4465fd/locations/us-central1/jobs/biosample---vienose--260415-210859-47-0-0
Launched job-id: biosample---vienose--260415-210859-47
To check the status, run:
  dstat --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'biosample---vienose--260415-210859-47' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'biosample---vienose--260415-210859-47' --users 'vienose'


In [7]:
%%bash --out saige_gene_step02

LOWER=4
UPPER=5
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bed" \
  --input plink_bim="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bim" \
  --input plink_fam="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/age_tests/" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/biosample_age_single_assoc.sh"

done

Job properties:
  job-id: biosample---vienose--260415-210901-67
  job-name: biosample-age-single-assoc
  user-id: vienose
Provider internal-id (operation): projects/terra-vpc-sc-fc4465fd/locations/us-central1/jobs/biosample---vienose--260415-210901-67-0-0
Launched job-id: biosample---vienose--260415-210901-67
To check the status, run:
  dstat --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'biosample---vienose--260415-210901-67' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'biosample---vienose--260415-210901-67' --users 'vienose'


In [8]:
%%bash --out saige_gene_step02

LOWER=15
UPPER=16
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bed" \
  --input plink_bim="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bim" \
  --input plink_fam="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/age_tests/" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/biosample_age_single_assoc.sh"

done

Job properties:
  job-id: biosample---vienose--260415-210903-44
  job-name: biosample-age-single-assoc
  user-id: vienose
Provider internal-id (operation): projects/terra-vpc-sc-fc4465fd/locations/us-central1/jobs/biosample---vienose--260415-210903-44-0-0
Launched job-id: biosample---vienose--260415-210903-44
To check the status, run:
  dstat --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'biosample---vienose--260415-210903-44' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'biosample---vienose--260415-210903-44' --users 'vienose'


In [9]:
%%bash --out saige_gene_step02

LOWER=17
UPPER=18
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bed" \
  --input plink_bim="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bim" \
  --input plink_fam="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/age_tests/" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/biosample_age_single_assoc.sh"

done

Job properties:
  job-id: biosample---vienose--260415-210905-18
  job-name: biosample-age-single-assoc
  user-id: vienose
Provider internal-id (operation): projects/terra-vpc-sc-fc4465fd/locations/us-central1/jobs/biosample---vienose--260415-210905-18-0-0
Launched job-id: biosample---vienose--260415-210905-18
To check the status, run:
  dstat --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'biosample---vienose--260415-210905-18' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'biosample---vienose--260415-210905-18' --users 'vienose'


In [3]:
!dstat --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs --users 'vienose' --status '*'

Job Name         Status    Last Update
---------------  --------  --------------
biosample-ag...  SUCCESS   04-15 21:12:40
biosample-ag...  SUCCESS   04-15 21:12:27
biosample-ag...  SUCCESS   04-15 21:12:47
biosample-ag...  SUCCESS   04-15 21:12:37
high-confide...  CANCELED  04-15 13:12:34
high-confide...  SUCCESS   04-15 15:05:03
high-confide...  SUCCESS   04-15 17:22:05
high-confide...  SUCCESS   04-15 14:26:43
high-confide...  SUCCESS   04-15 16:42:09
high-confide...  SUCCESS   04-15 16:03:07
high-confide...  SUCCESS   04-15 15:38:24
high-confide...  SUCCESS   04-15 15:22:40
high-confide...  SUCCESS   04-15 14:33:43
high-confide...  SUCCESS   04-15 16:26:35
high-confide...  SUCCESS   04-15 16:45:32
high-confide...  SUCCESS   04-15 15:42:33
high-confide...  SUCCESS   04-15 15:44:39
high-confide...  SUCCESS   04-15 15:35:57
high-confide...  SUCCESS   04-15 16:08:54
high-confide...  SUCCESS   04-15 16:15:52
high-confide...  SUCCESS   04-15 16:05:20
high-confide...  SUCCESS   04-15 15:4

## DOWNLOAD RESULTS FROM THE WORKSPACE BUCKET 

In [2]:
!rm -rf saige_gene_step02
!rm -rf merged_saige_gene_step02
!rm -rf markerList

In [3]:
!mkdir -p saige_gene_step02
!mkdir -p merged_saige_gene_step02
!mkdir -p markerList 

In [4]:
!gsutil cp {my_bucket}/data/saige_step02/LOVO_analysis/* saige_gene_step02
!gsutil cp {my_bucket}/data/saige_step02/HC_marklist/* markerList

Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355236:C:T...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355236:C:T.index...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355236:C:T.markerList.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355236:C:T.singleAssoc.txt...
- [4 files][  7.0 KiB/  7.0 KiB]                                                
==> NOTE: You are performing a sequence of gsutil operations that may
run significantly faster if you instead use gsutil -m cp ... Please
see the -m section under "gsutil help options" for further information
about when gsutil -m can be advantageous.

Copying gs://fc-se

Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355320:C:T.singleAssoc.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355320:C:T.singleAssoc.txt_temp...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355323:G:T...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355323:G:T.index...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355323:G:T.markerList.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355323:G:T.singleAssoc.txt...
Copying 

Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355389:C:T.index...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355389:C:T.markerList.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355389:C:T.singleAssoc.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355389:C:T.singleAssoc.txt_temp...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355392:C:G...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355392:C:G.index...
Copying gs://fc-se

Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355431:G:C.index...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355431:G:C.markerList.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355431:G:C.singleAssoc.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355431:G:C.singleAssoc.txt_temp...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355458:C:T...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355458:C:T.index...
Copying gs://fc-se

Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355506:G:A.index...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355506:G:A.markerList.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355506:G:A.singleAssoc.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355506:G:A.singleAssoc.txt_temp...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355511:A:G...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47355511:A:G.index...
Copying gs://fc-se

Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47358887:C:T.index...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47358887:C:T.markerList.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47358887:C:T.singleAssoc.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47358887:C:T.singleAssoc.txt_temp...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47358890:C:T...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr11:47358890:C:T.index...
Copying gs://fc-se

... (344 outputs left out)

Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276301:G:T.markerList.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276301:G:T.singleAssoc.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276301:G:T.singleAssoc.txt_temp...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276301:G:T_deprivation_index...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276301:G:T_deprivation_index.index...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:10527

Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276408:T:C_deprivation_index.index...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276408:T:C_deprivation_index.markerList.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276408:T:C_deprivation_index.singleAssoc.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276408:T:C_deprivation_index.singleAssoc.txt_temp...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276413:T:TC...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phen

Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276463:A:C...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276463:A:C.index...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276463:A:C.markerList.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276463:A:C.singleAssoc.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276463:A:C.singleAssoc.txt_temp...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/LOVO_analysis/EUR_aml_phenotype_lovo_high_confidence_chr4:105276463:A:C_deprivation_index...
Copyin

Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/HC_marklist/EUR-AML-Ultra-Rare-HGVS.markerList.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/HC_marklist/EUR-AML-Ultra-Rare-Variants-Deprivation-Index.markerList.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/HC_marklist/EUR-AML-Ultra-Rare-Variants.markerList.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/HC_marklist/GDC-frequent-mutations-deprivation-index.tsv...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/HC_marklist/GDC-frequent-mutations.tsv...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/HC_marklist/Group-Annotation.txt...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/HC_marklist/IDH2-frequent-mutations.2025-11-08.tsv...
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_ste

In [5]:
# Load packages 
import os
import glob
import pandas as pd

# ===============================================================================================
# Load files from asa
# ===============================================================================================

files = glob.glob("saige_gene_step02/*_deprivation_index")
files = [f for f in files if ".index" not in f]
files = [f for f in files if ".singleAssoc" not in f]
files = [f for f in files if ".markerList" not in f]
files =[]
dfs = []
for f in files:
    df = pd.read_csv(f, sep="\t", header=0)  
    df['filename'] = os.path.basename(f)    
    dfs.append(df)


combined_df = pd.concat(dfs, ignore_index=True)
combined_df["var"] = combined_df["filename"].str.split("_", expand = True)[6]
combined_df = combined_df.drop(columns = ['filename'])

# ================================================================================================
# Filter out variants not present in the rare and ultra-rare marker list files 
# ================================================================================================

ultra_rare_variants = pd.read_csv("markerList/EUR-AML-Ultra-Rare-Variants.markerList.txt", 
                                  sep = '\t')
rare_variants = pd.read_csv("markerList/EUR-AML-Rare-Variants.markerList.txt", 
                            sep = '\t')
combined_df = combined_df[(combined_df['var'].isin(ultra_rare_variants['Ultra_Rare_Variants'])) | 
                          (combined_df['var'].isin(rare_variants['Rare_Variants']))] 

# ================================================================================================
# Merge annotation file with leave-one-variant-out file 
# ================================================================================================

group_annotation = pd.read_csv("markerList/Group-Annotation.txt", sep = '\t')
combined_df = group_annotation.merge(combined_df, how = "inner", on =["var", 'Region'])

# ================================================================================================
# Merge all rare and ultra-rare AoU variants identified in Genomic Data Commons into a single list
# ================================================================================================

gdc_mutations = pd.read_csv("markerList/GDC-frequent-mutations.tsv", 
                            sep = '\t')
hgvs_ultra_rare = pd.read_csv("markerList/EUR-AML-Ultra-Rare-HGVS.markerList.txt", 
                              sep = '\t')
hgvs_rare = pd.read_csv("markerList/EUR-AML-Rare-HGVS.markerList.txt", 
                        sep = '\t')

hgvs_variants = pd.concat([hgvs_ultra_rare, hgvs_rare])
hgvs_variants["Variants"] = hgvs_variants.apply(
    lambda row: row["Ultra_Rare_Variants"] 
                if pd.notna(row["Ultra_Rare_Variants"]) 
                else row["Rare_Variants"],
    axis=1
) 

hgvs_variants = (
    hgvs_variants.merge(
        gdc_mutations.rename(columns={'dna_change':'HGVS'}), 
                    on = "HGVS", 
        how ="inner" )
)

# ================================================================================================
# Merge all rare and ultra-rare AoU variants identified in ClinVar into a single list
# ================================================================================================

clinvar_ultra_rare = pd.read_csv("markerList/EUR-AML-Ultra-Rare-Clinvar.markerList.txt", 
                                 sep = '\t')
clinvar_rare = pd.read_csv("markerList/EUR-AML-Rare-Clinvar.markerList.txt", 
                           sep = '\t')
clinvar_variants = pd.concat([clinvar_ultra_rare, clinvar_rare])

# ================================================================================================
# Label variants as either present or absent in the Genomic Data Commons or Clinvar 
# ================================================================================================

combined_df['Clinvar_Match'] = combined_df['var'].isin(clinvar_variants['Variants']).map(
    {True: "Match", False: "No Match"}
)
combined_df['GDC_Match'] = combined_df['var'].isin(hgvs_variants['Variants']).map(
    {True: "Match", False: "No Match"}
)
all_variants = set(clinvar_variants['Variants']) | set(hgvs_variants['Variants'])

combined_df['Match_Status'] = combined_df['var'].isin(all_variants).map(
    {True: "Match", False: "No Match"}
)

combined_df = combined_df[['var','anno','Region','Group','max_MAF',
                'Pvalue','Pvalue_Burden','Pvalue_SKAT',
                'BETA_Burden','SE_Burden',
                'Clinvar_Match','GDC_Match','Match_Status']]
# Save results 
combined_df.to_csv("EUR_aml_phenotype_leave_one_variant_out.txt", 
                   sep="\t", index=False)

combined_df.to_csv(f"{my_bucket}/data/saige_step02/LOVO_marklist/EUR_aml_phenotype_leave_one_variant_out.txt", 
                   sep="\t", index=False)